# Session 14: Uncertainty and model selection

A school district reads a map of low-cost PM2.5 sensors, and a principal moves recess indoors when a sensor's corrected 24 hour mean reaches an AQI breakpoint: 35.5 µg/m³ (AQI 101, outdoor activity shortened or moved) or 55.5 µg/m³ (AQI 151, red, cancelled or moved). Someone chooses the correction the map applies. This notebook chooses a correction by its error on data held out of the fit, estimates the correction's uncertainty by a bootstrap, and carries that uncertainty to the breakpoints. Section 1 works the method on Dataset A, the US-wide 24 hour collocation set, with Washington's sensors as the example. Section 2 applies it to your two sensors in the smoke set, with the bootstrap written by you. Section 3 compares two directions of fit on your sensors. Section 4 is the judgment cell.

The data are real, two datasets K. Barkjohn published on EPA ScienceHub for EPA's Office of Research and Development: the 24 hour set (2021), [doi:10.23719/1522388](https://doi.org/10.23719/1522388), and the smoke set (2023), [doi:10.23719/1528138](https://doi.org/10.23719/1528138), accessed 2026-11-05. The sensor readings came from PurpleAir and are used as EPA published them; the course does not copy them, so the first cell of section 1 downloads both sets into `data/`. `data/README.md` describes the files and who has which sensors.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 14 assignment on Canvas by 1:30 pm on Thursday, November 19. Submissions are individual. Every student downloads the same files, checked byte for byte, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart and holds your prompt record from section 2, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys` and `Path` from the standard library, NumPy, pandas and Matplotlib, and `curve_fit` from session 13, which fits every correction here. The bootstrap draws its resamples with `numpy.random`, met in session 8. Nothing new is imported. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Parameters

The data files; the two AQI breakpoints and the top of AQI 100; the reference level above which the raw-over-reference ratio is read; Dataset A's example state; the humidity at which the uncertainty is evaluated; the fewest hours in a valid day; the level from which a day's scatter is measured; the coverage of the band the map cannot call; the bootstrap's size and seed; section 3's plot limit; then your two sensors, set in section 2. Leave `SENSORS` as `CHANGE-ME` until section 2.

In [ ]:
DATA_DIR = Path("data")         # the two EPA sets, downloaded by data/fetch_barkjohn.py, and the assignment
TRAIN_FILE = DATA_DIR / "Data_DevelopmentUSPAcorrection_210408" / "Full24hrdataset.csv"       # Dataset A: 24 h means, sensor beside reference
SMOKE_FILE = DATA_DIR / "Dataset_PurpleAirSmoke_Barkjohn_1_18_23" / "Finaldataset_correcteddatasetwholding_10_26_22.csv"   # Dataset B: hourly, the smoke set
BREAKPOINTS = {"AQI 101": 35.5, "AQI 151": 55.5}   # ug/m3, 24 h PM2.5: recess shortened or moved at AQI 101, cancelled or moved at AQI 151 (EPA, since 2024-05-06)
MET_STANDARD = 35.4           # ug/m3, the top of AQI 100: a day at or below it met the 24 h standard
RATIO_ABOVE = 5               # ug/m3, the reference level above which section 1 reports raw over reference; below it the ratio is mostly noise
EXAMPLE_STATE = "WA"          # Dataset A's worked example: Washington's sensors, held out of the fit applied to them
RH_AT = 50                    # %, the humidity at which a breakpoint is turned back into a raw reading for the uncertainty
MIN_HOURS = 18                # hours of 24, 75%: the fewest for a valid 24 h mean
SCATTER_FROM = 20             # ug/m3, the corrected level from which the scatter of one day is measured, below the breakpoints
COVERAGE = 2                  # a day within COVERAGE sigma of a breakpoint is one the map cannot call
N_BOOT = 1000                 # resamples in each bootstrap
RNG_SEED = 427
PLOT_MAX_UG_M3 = 150          # ug/m3, the upper limit of section 3's plot, which shows the breakpoints rather than the worst smoke
SENSORS = ["CHANGE-ME", "CHANGE-ME"]   # your two sensors, from data/sensors.csv by your student number, section 2

## 1. Explore: choosing a correction, and how sure it is

Dataset A is EPA's 24 hour collocation set: each row is one day at one PurpleAir sensor beside a regulatory reference monitor, with the sensor's raw reading `PM25cf1`, the reference `PM25FM` and the sensor's relative humidity `RH`. The questions for this section are which correction to choose, and how well it is known. The first cell downloads both EPA sets into `data/` and checks them, which takes about a minute on campus Wi-Fi and a second after that. It then reads Dataset A, prints what it holds and how far the raw reading sits above the reference, and plots one against the other.

In [ ]:
%run data/fetch_barkjohn.py
train = pd.read_csv(TRAIN_FILE, usecols=["Date", "ID", "PM25FM", "PM25cf1", "RH", "PM25PARHcor"]).dropna(subset=["PM25FM", "PM25cf1", "RH"])
train["state"] = train["ID"].str.extract(r"^([A-Z]+)")[0]    # the state is the letters at the start of the sensor ID, AK1 -> AK
two_refs = train.duplicated(["ID", "Date"], keep=False)
clean = train["PM25FM"] > RATIO_ABOVE
print(f"Dataset A: {len(train)} rows, {train['ID'].nunique()} sensors in {train['state'].nunique()} states, {train['Date'].min()} to {train['Date'].max()}")
print(f"{two_refs.sum()} of the rows are {train[two_refs].groupby(['ID', 'Date']).ngroups} sensor-days beside two reference monitors, one row for each")
print(f"reference {train['PM25FM'].min():.1f} to {train['PM25FM'].max():.1f} ug/m3; raw {train['PM25cf1'].min():.1f} to {train['PM25cf1'].max():.1f} ug/m3; RH {train['RH'].min():.0f} to {train['RH'].max():.0f}%")
print(f"raw over reference, median of the {clean.sum()} rows with the reference above {RATIO_ABOVE} ug/m3: {(train.loc[clean, 'PM25cf1'] / train.loc[clean, 'PM25FM']).median():.2f}")
for label, level in BREAKPOINTS.items():
    print(f"rows at or above {label}, {level} ug/m3: reference {(train['PM25FM'] >= level).sum()}, raw {(train['PM25cf1'] >= level).sum()}")

fig, ax = plt.subplots()
ax.plot(train["PM25FM"], train["PM25cf1"], "o", color="black", markerfacecolor="none", markersize=3, alpha=0.3, label="one row, one sensor-day")
top = max(train["PM25FM"].max(), train["PM25cf1"].max())
ax.plot([0, top], [0, top], color="gray", linestyle="--", label="raw = reference")
ax.set_xlabel("reference PM2.5, 24 h mean (µg/m³)")
ax.set_ylabel("raw sensor PM2.5, 24 h mean (µg/m³)")
ax.set_title("Dataset A: 49 sensors beside reference monitors, 15 states")
ax.legend()
fig.tight_layout()
plt.show()

The 12,413 rows run up to 108.8 µg/m³ of reference and 256.3 µg/m³ of raw reading, and nearly every row above a few µg/m³ sits above the dashed raw = reference line, by a median factor of 1.53 on the 9,092 rows with the reference above 5 µg/m³, so the raw reading reaches AQI 101 on 664 rows where the reference does on 14.

Two corrections compete: the reference as a straight line in the raw reading, and the same line with a term in RH, since particles that take up water scatter more light. A fit's error on its own rows flatters it, because the fit was chosen to make those errors small. The test is the error on data held out of the fit, in groups, so the test is as new as the data the map will meet: a state's sensors share weather and smoke. The next cell fits both corrections to every state but one, fifteen times, prints each held-out state's mean absolute error, then the errors on the fit's own rows.

In [ ]:
def raw_only(raw, a, c):
    """Corrected PM2.5 (ug/m3) from the raw reading alone."""
    return a * raw + c


def raw_rh(X, a, b, c):
    """Corrected PM2.5 (ug/m3) from X = (raw reading in ug/m3, RH in %), stacked with np.vstack."""
    raw, rh = X
    return a * raw + b * rh + c


held_out = []
for state, held in train.groupby("state"):
    rest = train[train["state"] != state]
    p_raw, _ = curve_fit(raw_only, rest["PM25cf1"], rest["PM25FM"])
    p_rh, _ = curve_fit(raw_rh, np.vstack([rest["PM25cf1"], rest["RH"]]), rest["PM25FM"])
    held_out.append(pd.DataFrame({"state": state, "uncorrected": np.abs(held["PM25cf1"] - held["PM25FM"]),
                                  "raw only": np.abs(raw_only(held["PM25cf1"], *p_raw) - held["PM25FM"]),
                                  "raw + RH": np.abs(raw_rh(np.vstack([held["PM25cf1"], held["RH"]]), *p_rh) - held["PM25FM"])}))
held_out = pd.concat(held_out)
by_state = held_out.groupby("state").mean()
print("mean absolute error on the held-out state's rows (ug/m3), each state's fit made without it")
print(by_state.round(2).T.to_string())
print(f"over all held-out rows: uncorrected {held_out['uncorrected'].mean():.2f}, raw only {held_out['raw only'].mean():.2f}, raw + RH {held_out['raw + RH'].mean():.2f} ug/m3; "
      f"the RH term lowers the error in {(by_state['raw + RH'] < by_state['raw only']).sum()} of {len(by_state)} states")
p_raw_all, _ = curve_fit(raw_only, train["PM25cf1"], train["PM25FM"])
p_rh_all, _ = curve_fit(raw_rh, np.vstack([train["PM25cf1"], train["RH"]]), train["PM25FM"])
print(f"the same two fits made on all 15 states and judged on their own rows: raw only {np.abs(raw_only(train['PM25cf1'], *p_raw_all) - train['PM25FM']).mean():.2f}, "
      f"raw + RH {np.abs(raw_rh(np.vstack([train['PM25cf1'], train['RH']]), *p_rh_all) - train['PM25FM']).mean():.2f} ug/m3")

The RH term earns its place, so the map's correction is raw + RH fitted to all 15 states. The file carries a corrected column of its own, `PM25PARHcor`; fitting the same function to it recovers the coefficients behind it, a check on the fit, and they are not the same. Section 3 asks why. `curve_fit`'s covariance matrix `pcov` gives the standard error of a corrected value, as in session 13: the next cell turns each breakpoint back into the raw reading that corrects to it at `RH_AT`, and prints the standard error there.

In [ ]:
X = np.vstack([train["PM25cf1"], train["RH"]])
popt, pcov = curve_fit(raw_rh, X, train["PM25FM"])
fitted = raw_rh(X, *popt)
file_coef, _ = curve_fit(raw_rh, X, train["PM25PARHcor"])
print(f"the fit:               corrected = {popt[0]:.4f} x raw {popt[1]:+.4f} x RH {popt[2]:+.3f}")
print(f"behind PM25PARHcor:    corrected = {file_coef[0]:.4f} x raw {file_coef[1]:+.4f} x RH {file_coef[2]:+.3f}; "
      f"largest difference from the file's column {np.abs(raw_rh(X, *file_coef) - train['PM25PARHcor']).max():.1e} ug/m3")
at = {label: np.array([(level - popt[1] * RH_AT - popt[2]) / popt[0], RH_AT, 1.0]) for label, level in BREAKPOINTS.items()}   # (raw, RH, 1) at each breakpoint
for label, x in at.items():
    print(f"{label}: a raw reading of {x[0]:.1f} ug/m3 at RH {RH_AT}% corrects to {BREAKPOINTS[label]} ug/m3; standard error from pcov {np.sqrt(x @ pcov @ x):.2f} ug/m3")

`pcov`'s standard error assumes every row is an independent measurement with the same scatter. A **bootstrap** checks that by refitting to many resamples of the data, each drawn with replacement and as large as the data, and reading the spread of the refitted values. The next cell draws rows, the unit `pcov` treats as independent, refits the correction `N_BOOT` times, and prints the spread of a corrected value at each breakpoint beside `pcov`'s.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
row_coefs = np.empty((N_BOOT, 3))
for i in range(N_BOOT):
    sample = train.iloc[rng.integers(0, len(train), len(train))]     # rows, with replacement
    row_coefs[i], _ = curve_fit(raw_rh, np.vstack([sample["PM25cf1"], sample["RH"]]), sample["PM25FM"])
for label, x in at.items():
    print(f"{label}: SD of a corrected value from pcov {np.sqrt(x @ pcov @ x):.2f}, from resampling rows {(row_coefs @ x).std():.2f} ug/m3")

Washington's three sensors are the story's state, and none of them saw wildfire smoke in these years. The next cell defines `recess_table`, which counts, at each breakpoint, the days an estimate calls, the false alarms (called, but the reference was below), how many of those met the standard, and the misses (not called, but the reference was at or above). It refits the correction without Washington, applies it to Washington's rows, and prints the table for the raw reading and the correction.

In [ ]:
def recess_table(days, estimates, sigma=None):
    """Per estimate column and breakpoint: days called, reference at or above, false alarms, those that met the standard, misses;
    with sigma (a dict by breakpoint), also the days whose estimate lies within COVERAGE sigma of the breakpoint."""
    rows = []
    for est in estimates:
        for label, level in BREAKPOINTS.items():
            called, above = days[est] >= level, days["ref"] >= level
            row = {"estimate": est, "breakpoint": label, "called": int(called.sum()), "reference above": int(above.sum()),
                   "false alarms": int((called & ~above).sum()), "met the standard": int((called & (days["ref"] <= MET_STANDARD)).sum()),
                   "misses": int((~called & above).sum())}
            if sigma is not None:
                row[f"within {COVERAGE} sigma"] = int((np.abs(days[est] - level) < COVERAGE * sigma[label]).sum()) if est != "raw" else ""
            rows.append(row)
    return pd.DataFrame(rows)


wa = train[train["state"] == EXAMPLE_STATE].rename(columns={"PM25cf1": "raw", "PM25FM": "ref"})
not_wa = train[train["state"] != EXAMPLE_STATE]
p_not_wa, _ = curve_fit(raw_rh, np.vstack([not_wa["PM25cf1"], not_wa["RH"]]), not_wa["PM25FM"])
wa["corrected"] = raw_rh(np.vstack([wa["raw"], wa["RH"]]), *p_not_wa)
print(f"{EXAMPLE_STATE}: {len(wa)} rows at {wa['ID'].nunique()} sensors, {wa['Date'].min()} to {wa['Date'].max()}; reference maximum {wa['ref'].max():.1f} ug/m3")
print(recess_table(wa, ["raw", "corrected"]).to_string(index=False))

**Explain.** In two sentences: why does the error on held-out states, and not the error on the rows the fit was made from, choose between the two corrections, and what would a map that showed Washington's raw readings have cost its schools?

*Your answer:*

## 2. Apply: your two sensors

The smoke set is EPA's second collocation set: one row per sensor-hour, from 27 sensors between 2018 and 2020, many of them through the 2020 fire season, when the reference reached hundreds of µg/m³. Your two sensors differ: in most pairs one saw heavy smoke and one did not. Section 1's correction, fitted to all 15 states, carries over unchanged. What changes is the air, and how sure the map can be near a breakpoint.

**[TASK 1]** Find your student number in `data/sensors.csv` and set `SENSORS` in the Parameters cell to the two names beside it, for example `["Keeler_S", "RTP"]`. Run the Parameters cell, then the next cell. It reads the hourly file, about 3 seconds, keeps your two sensors' rows, and prints how many rows and how many distinct hours each holds, and the values of the file's two bookkeeping columns. Until `SENSORS` is set, it stops with `ValueError: no smoke-set rows for ['CHANGE-ME', 'CHANGE-ME']`.

In [ ]:
hourly = pd.read_csv(SMOKE_FILE, usecols=["hr", "city", "RH", "PA", "ref", "wholdtype", "adjtype"])
missing = [s for s in SENSORS if s not in set(hourly["city"])]
if missing:
    raise ValueError(f"no smoke-set rows for {missing}; set SENSORS in Parameters from data/sensors.csv")
hourly = hourly[hourly["city"].isin(SENSORS)]
for city, rows in hourly.groupby("city"):
    print(f"{city}: {len(rows)} rows, {rows['hr'].nunique()} distinct hours, {rows['hr'].min()[:10]} to {rows['hr'].max()[:10]}")
print(f"wholdtype holds {sorted(hourly['wholdtype'].unique())}; adjtype holds {sorted(hourly['adjtype'].unique())}")

**[TASK 2]** Each sensor-hour appears more than once in the file, once for each way the paper refitted its correction, which the two bookkeeping columns record, with the same `PA`, `ref` and `RH` each time. Replace `...` in the first line of the next cell with the rows of `hourly` that hold each sensor-hour once. The rest of the cell averages each day, stops with `AssertionError` if any day still holds more than 24 hours or if a sensor-hour was dropped, keeps days with at least `MIN_HOURS`, applies section 1's correction, prints each sensor's days and plots them. Until the line is filled, it stops with `AttributeError: 'ellipsis' object has no attribute 'assign'`.

In [ ]:
one_copy = ...   # [TASK 2] the rows of hourly that hold each sensor-hour once; the assert below checks it
one_copy = one_copy.assign(day=one_copy["hr"].str[:10])            # the timestamp cut to its date
mine = (one_copy.groupby(["city", "day"])
        .agg(hours=("PA", "size"), raw=("PA", "mean"), ref=("ref", "mean"), RH=("RH", "mean"))
        .reset_index())
assert mine["hours"].max() <= 24, f"a day holds {mine['hours'].max()} hours: each sensor-hour is not yet held once"
assert len(one_copy) == hourly.groupby("city")["hr"].nunique().sum(),     f"one_copy holds {len(one_copy)} rows, not one for each distinct hour TASK 1 printed for each sensor"
mine = mine[mine["hours"] >= MIN_HOURS].dropna().reset_index(drop=True)
mine["corrected"] = raw_rh(np.vstack([mine["raw"], mine["RH"]]), *popt)
for city, days in mine.groupby("city"):
    print(f"{city}: {len(days)} days with at least {MIN_HOURS} hours, {days['day'].min()} to {days['day'].max()}; reference maximum {days['ref'].max():.1f} ug/m3; "
          f"days with the reference at or above AQI 101: {(days['ref'] >= BREAKPOINTS['AQI 101']).sum()}, AQI 151: {(days['ref'] >= BREAKPOINTS['AQI 151']).sum()}")

fig, axes = plt.subplots(len(SENSORS), 1, figsize=(8, 6))
for ax, (city, days) in zip(axes, mine.groupby("city")):
    dates = pd.to_datetime(days["day"])
    ax.plot(dates, days["ref"], "o", color="black", markerfacecolor="none", markersize=3, label="reference")
    ax.plot(dates, days["corrected"], "s", color="gray", markerfacecolor="none", markersize=3, label="corrected sensor")
    for label, level in BREAKPOINTS.items():
        ax.axhline(level, color="gray", linestyle=":" if level < 50 else "--", label=label)
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(plt.ScalarFormatter())    # 10 and 100 written out, not as powers
    ax.yaxis.set_minor_formatter(plt.NullFormatter())
    ax.set_ylabel("PM2.5, 24 h mean (µg/m³)")
    ax.set_title(city)
    ax.tick_params(axis="x", labelrotation=20)
axes[-1].set_xlabel("date")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncols=4, fontsize="small")
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

*Write your one sentence description of the plot here.*

**Correct early, and clear the context**, today's AI practice, met in session 8. Before TASK 3, decide what each resample should draw, from the notes' rule, then ask the assistant in Claude Code for the cell in your own words: which notebook and task, what each draw takes, which names from section 1 the cell uses, and which name it must set, with its shape. If the first answer draws something other than what you decided, say so at once, in one sentence, before it writes more. Paste the code into the cell you add; if Claude Code asks to edit the notebook, decline. Send your prompt, and only then read the course's example of a first prompt:

`In sessions/S14-model-selection/activity.ipynb, section 2, TASK 3: bootstrap the forward correction raw_rh over <the unit I decided>. Draw N_BOOT resamples of those units in train with replacement, using np.random.default_rng(RNG_SEED), keep every row of each drawn unit, refit raw_rh to each resample with curve_fit, and stack the coefficients in boot_coefs, shape (N_BOOT, 3). Explain first why each draw takes that unit whole, then give the cell, and tell me how to check it against the given cell after it.`

Read the answer against the given cell after TASK 3, which prints your bootstrap's spread beside `pcov`'s and the rows'. Add a markdown cell under this one (hover just below this cell and click **+ Markdown**) holding these lines, and fill the first now:

```
**Prompt record**

- **My prompt:**
- **What the first answer drew, and the correction I gave, or none:**
- **What the given cell showed for my unit against rows:**
```

Fill the second and third lines after TASK 3. Your prompt record is one candidate for part (b) of the judgment cell, and the execution point needs it.

**[TASK 3]** In a code cell you add under this one (**+ Code**), bootstrap section 1's correction over the unit you decided: draw `N_BOOT` resamples of `train` with `np.random.default_rng(RNG_SEED)`, refit `raw_rh` to each with `curve_fit`, as section 1's cell does, and stack the fitted coefficients in `boot_coefs`, an array of shape (`N_BOOT`, 3). It takes about 7 seconds. The cell after it prints the spread of a corrected value at each breakpoint from `pcov`, from section 1's rows and from your bootstrap, the scatter of one day about the line, the two together as σ, and how many of your days lie within 2σ of each breakpoint. It stops with `NameError: name 'boot_coefs' is not defined` until your cell has run.

In [ ]:
assert np.shape(boot_coefs) == (N_BOOT, 3), f"boot_coefs has shape {np.shape(boot_coefs)}, not ({N_BOOT}, 3)"
resid = train["PM25FM"] - fitted
day_scatter = resid[fitted >= SCATTER_FROM].std()     # ug/m3, one day's scatter about the line, measured near the breakpoints
print(f"one day's scatter about the line: SD {resid.std():.2f} ug/m3 over all {len(resid)} rows, {day_scatter:.2f} over the {(fitted >= SCATTER_FROM).sum()} rows corrected to {SCATTER_FROM} ug/m3 or more")
sigma_at = {}
for label, x in at.items():
    line = (boot_coefs @ x).std()
    sigma_at[label] = np.hypot(line, day_scatter)       # the line's spread and one day's scatter, added in quadrature
    print(f"{label}: the line's SD from pcov {np.sqrt(x @ pcov @ x):.2f}, resampling rows {(row_coefs @ x).std():.2f}, your bootstrap {line:.2f} ug/m3; "
          f"with one day's scatter, sigma = {sigma_at[label]:.2f} ug/m3")
for city, days in mine.groupby("city"):
    near = {label: int((np.abs(days["corrected"] - BREAKPOINTS[label]) < COVERAGE * sigma_at[label]).sum()) for label in BREAKPOINTS}
    print(f"{city}: of {len(days)} days, within {COVERAGE} sigma of AQI 101: {near['AQI 101']}; of AQI 151: {near['AQI 151']}")

**Explain.** Before you compare: in two sentences, give your σ at AQI 151, the unit your bootstrap drew and its SD beside the rows', and how many of your days lie within 2σ of each breakpoint. Then compare with a neighbor: whose sensors put more days beyond the map's reach, and if your units differed, did the unit move σ? Add one sentence on where your two answers differ and what in your data explains it.

*Your answer:*

## 3. Analyze: which way to fit the correction

Section 1 fitted the reference on the raw reading and RH, ordinary least squares with the reference as y: call it fit A, the forward fit. The file's `PM25PARHcor` comes from fit B: the raw reading fitted on the reference and RH, then solved for the reference, which is how the paper fitted it. Both are defensible. A forward fit predicts the reference best on average, and an inverted fit keeps the sensor's own response, which a noisy x would otherwise flatten. The question for this section is which errors each makes at your sensors. The next cell fits B, prints both fits beside the file's, applies both to your days, and prints the recess table for raw, A and B at each breakpoint, sensor by sensor and together, with the days within 2σ. It plots each corrected value against the reference near the breakpoints.

In [ ]:
(slope_b, rh_b, intercept_b), _ = curve_fit(raw_rh, np.vstack([train["PM25FM"], train["RH"]]), train["PM25cf1"])   # raw = s ref + r RH + c
popt_b = np.array([1 / slope_b, -rh_b / slope_b, -intercept_b / slope_b])                                     # solved for the reference
print(f"fit A, reference on raw and RH:            corrected = {popt[0]:.4f} x raw {popt[1]:+.4f} x RH {popt[2]:+.3f}")
print(f"fit B, raw on reference and RH, inverted:  corrected = {popt_b[0]:.4f} x raw {popt_b[1]:+.4f} x RH {popt_b[2]:+.3f}")
print(f"behind the file's PM25PARHcor:              corrected = {file_coef[0]:.4f} x raw {file_coef[1]:+.4f} x RH {file_coef[2]:+.3f}")
mine["fit A"] = mine["corrected"]
mine["fit B"] = raw_rh(np.vstack([mine["raw"], mine["RH"]]), *popt_b)
tables = []
for name, days in [*mine.groupby("city"), ("both", mine)]:
    table = recess_table(days, ["raw", "fit A", "fit B"], sigma_at)
    table.insert(0, "sensors", name)
    tables.append(table)
tables = pd.concat(tables)
for label, level in BREAKPOINTS.items():
    print(f"\n{label}, {level} ug/m3; sigma {sigma_at[label]:.2f} ug/m3")
    print(tables[tables["breakpoint"] == label].drop(columns="breakpoint").to_string(index=False))

fig, ax = plt.subplots()
ax.plot(mine["ref"], mine["fit A"], "o", color="black", markerfacecolor="none", markersize=4, label="fit A, forward")
ax.plot(mine["ref"], mine["fit B"], "^", color="gray", markerfacecolor="none", markersize=4, label="fit B, inverted")
ax.plot([0, PLOT_MAX_UG_M3], [0, PLOT_MAX_UG_M3], color="gray", linestyle="-", linewidth=0.8, label="corrected = reference")
for label, level in BREAKPOINTS.items():
    ax.axhline(level, color="gray", linestyle=":" if level < 50 else "--", label=label)
    ax.axvline(level, color="gray", linestyle=":" if level < 50 else "--")
ax.set_xlim(0, PLOT_MAX_UG_M3)
ax.set_ylim(0, PLOT_MAX_UG_M3)
ax.set_xlabel("reference PM2.5, 24 h mean (µg/m³)")
ax.set_ylabel("corrected sensor PM2.5, 24 h mean (µg/m³)")
ax.set_title(f"your two sensors, {len(mine)} days; {(mine['ref'] > PLOT_MAX_UG_M3).sum()} days above {PLOT_MAX_UG_M3} µg/m³ not shown")
ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**Explain.** In two or three sentences, say which error each fit makes at your sensors, at each breakpoint where the two differ: which fit gives false alarms, and how many of those fell on days that met the standard, and which gives misses. Say which error a principal can less afford, and how many of your days lie within 2σ, where no fit can say. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): as the analyst choosing the map's correction, which fit should the district use? Cite, at your two sensors, the raw reading's false alarms and your days within 2σ at both breakpoints, and, at each breakpoint where the fits differ, each fit's false alarms, with those on days that met the standard, and misses, and say what the choice costs each way: children kept indoors on days the air met the standard, or children outside in smoke. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your data. Your prompt record from section 2 is the natural source; the reason has to cite a value from your output, not a general rule. This session's AI practice, named in the notes, is *correct early, and clear the context*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"SciPy      {scipy.__version__}")
print(f"Matplotlib {matplotlib.__version__}")